8.Develop a language model to predict next best word.

1. import libraries

In [1]:
# Import required libraries for text processing and deep learning model building.
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

2. DATA COLLECTION


In [2]:
# Small training text corpus used to teach next-word patterns.
data = """Deep learning is powerful
Deep learning is fun
Machine learning is amazing
Artificial intelligence is the future
I love learning new things
Data science is interesting
Neural networks are powerful
AI is transforming the world"""

3. TEXT PREPROCESSING

In [3]:
# Convert text into tokens (word IDs) so model can read numbers, not raw words.
# Convert to lowercase
data = data.lower()

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

total_words = len(tokenizer.word_index) + 1
print("Total Words:", total_words)

Total Words: 25


4. Create Training Sequences (N-grams)

In [4]:
# Build n-gram style sequences: each sequence helps predict its last word.
input_sequences = []

for line in data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

print(input_sequences[:5])

[[3, 2], [3, 2, 1], [3, 2, 1, 4], [3, 2], [3, 2, 1]]


5. PADDING

In [5]:
# Pad shorter sequences with zeros on left so all inputs have same length.
max_seq_len = max([len(seq) for seq in input_sequences])

input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

print("Max Sequence Length:", max_seq_len)

Max Sequence Length: 5


6: Split Data (X and y)

In [6]:
# X = input words, y = next word label for each sequence.
# One-hot converts each y word ID into a probability target vector.
input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encoding
y = to_categorical(y, num_classes=total_words)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (27, 4)
Shape of y: (27, 25)


7: Model Building

In [7]:
# Embedding learns word vectors, LSTM learns word order context, Dense predicts next word.
model = Sequential()

model.add(Embedding(total_words, 10, input_length=max_seq_len-1))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)



```
8 . MODEL TRAINING
```



In [8]:
# Train model for multiple epochs so it gradually learns better next-word predictions.
history = model.fit(X, y, epochs=100, verbose=1)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0000e+00 - loss: 3.2193
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.1481 - loss: 3.2158
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.1852 - loss: 3.2123
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.2593 - loss: 3.2088
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.2593 - loss: 3.2051
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.2593 - loss: 3.2012
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.2222 - loss: 3.1971
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.2222 - loss: 3.1926
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.2222 - loss: 3.1878
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.2222 - loss: 3.1825
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.2222 - loss: 3.1767
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.2222 - 

9: Prediction Function

In [9]:
# Start from seed text, predict one word, append it, and repeat.
def predict_next_word(seed_text, next_words=1):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

10: Testing / Prediction

In [10]:
# Try a few seed phrases to see generated next words.
print(predict_next_word("deep learning is", 2))
print(predict_next_word("machine learning", 2))
print(predict_next_word("artificial intelligence", 2))

deep learning is powerful world
machine learning is the
artificial intelligence is the


11: Evaluation

In [11]:
# Evaluate on training data to get final loss and accuracy numbers.
loss, accuracy = model.evaluate(X, y, verbose=0)
print("Model Accuracy:", accuracy)

Model Accuracy: 0.5555555820465088
